# GourNet v2 — Background-Robust Fix (same architecture)

Fix notebook based on `enhanced-gournetv2-12-mseed.ipynb`. Upload to Kaggle alongside the same MangoLeafBD mirror dataset.

**What was wrong (evidence, not speculation):**
- Baseline split used `holdout.take() / holdout.skip()` on an independently reshuffled `holdout_ds`, so val/test membership is not guaranteed disjoint. Caching afterwards does not fix it.
- Old Grad-CAM differentiated softmax probabilities at `Block4_Conv` (pre-GroupNorm). On ~99% confident images the probability map collapses (Anthracnose returned an all-zero map) and the pre-norm layer distorts localization. App now uses pre-softmax logits at `Block4_ReLU`.
- Crop test: Healthy (99.04%) -> tightly-cropped leaf predicts Powdery Mildew (99.42%). Crop changes scale too, so this is a robustness warning, not proof — but CuttingWeevil maps also keep background tint.

**What this notebook changes (architecture untouched):**
1. File-path stratified 80/10/10 splits with a saved manifest CSV; val/test provably disjoint.
2. Auto leaf masks (`rembg` U²-Net, classical Otsu fallback) with QC — never a green-pixel filter, which would delete brown/yellow/black lesions.
3. Class-independent background randomization on 50% of training images (solid paper-tone backgrounds + noise); originals kept for the other 50%. Val/test never composited.
4. The `RandomZoom` + `RandomContrast` layers the baseline intro claimed but its code lacked.
5. Robustness eval per seed: original test acc, background-swapped acc drop, background-only acc (≈chance means no background shortcut), leaf-only acc, plus logit `Block4_ReLU` Grad-CAM.


## 1. Imports & config


In [ ]:
import os
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

print('TensorFlow:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 3
LEARNING_RATE = 0.001
BG_SWAP_PROB = 0.5  # fraction of TRAIN images composited onto randomized backgrounds
MASK_DIRNAME = 'leaf_masks_robust'

CLASS_NAMES = [
    "Anthracnose",
    "Bacterial Canker",
    "Cutting Weevil",
    "Die Back",
    "Gall Midge",
    "Healthy",
    "Powdery Mildew",
    "Sooty Mould",
]

SEEDS = [123, 42, 7, 2024, 2026, 55, 777, 2025, 314, 8080, 999, 31415]


### Locate the dataset folder

Same search logic as the baseline: find the directory under `/kaggle/input` that directly contains all 8 class subfolders.


In [ ]:
KAGGLE_INPUT = pathlib.Path('/kaggle/input')
WORKING = pathlib.Path('/kaggle/working')


def find_dataset_dir(root: pathlib.Path, expected_classes: list) -> pathlib.Path:
    expected_lower = {c.lower() for c in expected_classes}
    candidates = []
    for dirpath, dirnames, _ in os.walk(root):
        if expected_lower.issubset({d.lower() for d in dirnames}):
            candidates.append(pathlib.Path(dirpath))
    if not candidates:
        raise FileNotFoundError(
            f'No folder under {root} contains all of: {expected_classes}. '
            'Check the Kaggle Data panel and set DATA_DIR manually.'
        )
    candidates.sort(key=lambda p: len(p.parts))
    return candidates[0]


DATA_DIR = find_dataset_dir(KAGGLE_INPUT, CLASS_NAMES)
print('Using dataset directory:', DATA_DIR)
print('Contents:', sorted(os.listdir(DATA_DIR)))


## 2. Fixed file-path splits (replaces `take()` / `skip()`)

Why: the baseline calls `image_dataset_from_directory` twice then bisects `holdout_ds` with `.take(n) // .skip(n)`. Each pipeline reshuffles independently, so the val/test bisection is not guaranteed disjoint. Fix: split **file paths** with stratification, write a manifest CSV, build `tf.data` from paths. Shuffle train only.


In [ ]:
from sklearn.model_selection import train_test_split


def list_image_files(data_dir: pathlib.Path):
    exts = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
    rows = []
    for cls in CLASS_NAMES:
        for p in sorted((data_dir / cls).rglob('*')):
            if p.is_file() and p.suffix.lower() in exts:
                rows.append((str(p), cls))
    return pd.DataFrame(rows, columns=['filepath', 'class'])


def stratified_file_split(files_df: pd.DataFrame, seed: int):
    """Deterministic stratified 80/10/10 split on file paths."""
    train_df, holdout_df = train_test_split(
        files_df, test_size=0.2, random_state=seed, stratify=files_df['class']
    )
    val_df, test_df = train_test_split(
        holdout_df, test_size=0.5, random_state=seed, stratify=holdout_df['class']
    )
    for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        df = df.copy()
        df['split'] = name
        df['seed'] = seed
        yield df


FILES_DF = list_image_files(DATA_DIR)
print('Total images:', len(FILES_DF))
print(FILES_DF['class'].value_counts().to_string())

train_df, val_df, test_df = stratified_file_split(FILES_DF, SEED)
assert set(train_df.filepath).isdisjoint(val_df.filepath)
assert set(train_df.filepath).isdisjoint(test_df.filepath)
assert set(val_df.filepath).isdisjoint(test_df.filepath)
manifest = pd.concat([train_df, val_df, test_df], ignore_index=True)
manifest.to_csv(WORKING / f'split_manifest_seed{SEED}.csv', index=False)
print(manifest.groupby(['split', 'class']).size().unstack(fill_value=0))
print('Manifest saved — val/test provably disjoint.')


### Path-based `tf.data` pipelines (raw 0–255 in, model rescales internally)


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}


def _decode_image(path: tf.Tensor):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE, method='bilinear')
    return tf.cast(img, tf.float32)  # 0-255; model Rescaling handles /255


def ds_from_df(df: pd.DataFrame, shuffle: bool, seed: int):
    ds = tf.data.Dataset.from_tensor_slices(
        (df['filepath'].to_numpy(), df['class'].map(CLASS_TO_IDX).to_numpy().astype(np.int64))
    )
    if shuffle:
        ds = ds.shuffle(len(df), seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: (_decode_image(p), y), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)
    return ds


val_ds = ds_from_df(val_df, shuffle=False, seed=SEED)
test_ds = ds_from_df(test_df, shuffle=False, seed=SEED)
class_names = CLASS_NAMES
print('val batches:', tf.data.experimental.cardinality(val_ds).numpy(),
      '| test batches:', tf.data.experimental.cardinality(test_ds).numpy())


## 3. Auto leaf masks (no green filter — lesions are preserved)

Tries `rembg` (U²-Net general foreground). Falls back to Otsu + largest-component if `rembg` or its weights are unavailable. Masks are computed once for all files and reused across seeds. Review the QC grid before training: coverage should mostly sit in 5–90%; rejected masks fall back to the original image (never a zeroed leaf).


In [ ]:
# Run once: needs internet the first time to fetch the u2net weights.
# !pip install -q rembg onnxruntime

import io
from PIL import Image

MASK_DIR = WORKING / MASK_DIRNAME
MASK_DIR.mkdir(parents=True, exist_ok=True)


def _rembg_mask(pil_img: Image.Image):
    from rembg import new_session, remove
    global _REMBG_SESSION
    try:
        _REMBG_SESSION
    except NameError:
        _REMBG_SESSION = new_session('u2net')
    out = remove(pil_img, session=_REMBG_SESSION)
    if out.mode == 'RGBA':
        return np.array(out.split()[-1])
    return None


def _otsu_fallback_mask(pil_img: Image.Image):
    g = np.array(pil_img.convert('L'))
    blur = tf.image.resize(g[None, ..., None].astype(np.float32), (56, 56)).numpy()[0, ..., 0]
    hist, _ = np.histogram(blur, bins=256, range=(0, 255))
    p = hist / hist.sum()
    omega = np.cumsum(p)
    mu = np.cumsum(p * np.arange(256))
    mu_t = mu[-1]
    sigma = (mu_t * omega - mu) ** 2 / np.maximum(omega * (1 - omega), 1e-9)
    t = float(np.argmax(sigma))
    small = (blur < t).astype(np.uint8)
    # largest 4-connected dark component = leaf candidate
    seen = np.zeros_like(small, bool)
    best = np.zeros_like(small, bool)
    for ys, xs in zip(*np.nonzero(small)):
        if seen[ys, xs]:
            continue
        stack, comp = [(ys, xs)], []
        seen[ys, xs] = True
        while stack:
            y, x = stack.pop()
            comp.append((y, x))
            for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            
                ny, nx = y + dy, x + dx
                if 0 <= ny < 56 and 0 <= nx < 56 and small[ny, nx] and not seen[ny, nx]:
                    seen[ny, nx] = True
                    stack.append((ny, nx))
        if len(comp) > best.sum():
            best = np.zeros_like(small, bool)
            for y, x in comp:
                best[y, x] = True
    m = Image.fromarray((best * 255).astype(np.uint8)).resize(pil_img.size, Image.BILINEAR)
    return np.array(m)


def mask_for_file(filepath: str):
    """Returns (mask_path, coverage, method). Whole-leaf foreground, not green-only."""
    stem = pathlib.Path(filepath).stem + '.png'
    # disambiguate duplicate stems across classes
    cls = pathlib.Path(filepath).parent.name
    out = MASK_DIR / f'{cls}__{stem}'
    if out.exists():
        m = np.array(Image.open(out).convert('L'))
        return str(out), float((m > 127).mean()), 'cached'
    pil_img = Image.open(filepath).convert('RGB')
    method, raw = 'otsu-fallback', None
    try:
        raw = _rembg_mask(pil_img)
        method = 'rembg-u2net' if raw is not None else method
    except Exception as e:
        print(f'rembg unavailable ({type(e).__name__}), using Otsu fallback.')
    if raw is None:
        raw = _otsu_fallback_mask(pil_img)
    m = Image.fromarray(raw).resize((IMG_SIZE[1], IMG_SIZE[0]), Image.BILINEAR)
    m = np.array(m)
    cov = float((m > 127).mean())
    if not (0.05 <= cov <= 0.90):
        # reject: fall back to full-frame (original image), never blank the leaf
        m = np.full_like(m, 255)
        method += '+rejected-fullframe'
        cov = 1.0
    Image.fromarray(m).save(out)
    return str(out), cov, method


# Compute once (subset preview first, then full run). Comment out after first run if re-running.
preview = FILES_DF.sample(min(9, len(FILES_DF)), random_state=SEED)
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for ax, (_, row) in zip(axes.flat, preview.iterrows()):
    mp, cov, method = mask_for_file(row['filepath'])
    ax.imshow(np.array(Image.open(row['filepath']).convert('RGB').resize((224, 224))))
    ax.imshow(np.array(Image.open(mp).convert('L').resize((224, 224))) > 127, alpha=0.35, cmap='jet')
    ax.set_title(f"{row['class'][:12]} cov={cov:.2f}\n{method}", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig(WORKING / 'mask_qc_preview.png', dpi=130)
plt.show()


In [ ]:
# Full mask pass over every file (reused by all 12 seeds). Skips cached masks.
qc_rows = []
for _, row in FILES_DF.iterrows():
    mp, cov, method = mask_for_file(row['filepath'])
    qc_rows.append({'filepath': row['filepath'], 'class': row['class'],
                   'mask_path': mp, 'coverage': cov, 'method': method})
qc_df = pd.DataFrame(qc_rows)
qc_df.to_csv(WORKING / 'mask_qc.csv', index=False)
print(qc_df.groupby('method').size().to_string())
print('coverage describe:\n', qc_df['coverage'].describe().to_string())
print('Review mask_qc_preview.png before training. Rejected masks use the original image.')


## 4. Class-independent background randomization (train only)

Each training leaf is composited with probability 0.5 onto a randomized paper-tone background sampled **independent of the label**. Val/test are never composited. Backgrounds are synthetic (no second leaf can leak in from another photo).


In [ ]:
MASK_LOOKUP = dict(zip(qc_df['filepath'], qc_df['mask_path']))


def _random_paper_background(h, w, rng):
    base = rng.uniform(205, 248, size=(3,)).astype(np.float32)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    grad = ((xx / max(w - 1, 1) - 0.5) * rng.uniform(-14, 14)
            + (yy / max(h - 1, 1) - 0.5) * rng.uniform(-14, 14))
    bg = base[None, None, :] + grad[..., None]
    bg += rng.normal(0, rng.uniform(1.5, 5.0), size=(h, w, 1)).astype(np.float32)
    return np.clip(bg, 0, 255)


def _load_train_sample(filepath, label, seed_pair):
    # tf.numpy_function passes strings as bytes (b'...'); decode to str for dict lookup.
    if isinstance(filepath, (bytes, bytearray)):
        filepath = filepath.decode('utf-8')
    else:
        try:
            filepath = bytes(filepath).decode('utf-8')
        except Exception:
            filepath = str(filepath)
    rng = np.random.default_rng([int(seed_pair[0]), int(seed_pair[1])])
    img = np.array(Image.open(filepath).convert('RGB').resize((IMG_SIZE[1], IMG_SIZE[0])), dtype=np.float32)
    if rng.random() < BG_SWAP_PROB:
        m = np.array(Image.open(MASK_LOOKUP[filepath]).convert('L').resize(
            (IMG_SIZE[1], IMG_SIZE[0])), dtype=np.float32) / 255.0
        m = m[..., None]
        bg = _random_paper_background(IMG_SIZE[0], IMG_SIZE[1], rng)
        img = m * img + (1.0 - m) * bg
    return img.astype(np.float32), np.int64(int(label))


def train_ds_from_df(df: pd.DataFrame, seed: int):
    paths = df['filepath'].to_numpy()
    labels = df['class'].map(CLASS_TO_IDX).to_numpy().astype(np.int64)
    counter = tf.data.Dataset.range(len(df))
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.shuffle(len(df), seed=seed, reshuffle_each_iteration=True)
    ds = ds.enumerate()

    def _map(i, pv):
        p, y = pv
        img, lab = tf.numpy_function(
            _load_train_sample, [p, y, tf.stack([tf.cast(seed, tf.int64), tf.cast(i, tf.int64)])],
            [tf.float32, tf.int64])
        img.set_shape((*IMG_SIZE, 3))
        lab.set_shape([])
        return img, lab

    ds = ds.map(_map, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)


train_ds = train_ds_from_df(train_df, SEED)
plt.figure(figsize=(10, 4))
for images, labels in train_ds.take(1):
    for i in range(min(8, images.shape[0])):
        ax = plt.subplot(2, 4, i + 1)
        ax.imshow(images[i].numpy().astype('uint8'))
        ax.set_title(class_names[int(labels[i])][:12])
        ax.axis('off')
plt.suptitle('Train preview: ~50% randomized paper backgrounds, 50% originals')
plt.tight_layout()
plt.savefig(WORKING / 'train_bg_preview.png', dpi=130)
plt.show()


## 5. Build GourNet v2 — architecture identical to baseline


In [ ]:
def build_gournet_v2(num_classes: int = 8, input_shape=(224, 224, 3)) -> tf.keras.Model:
    """EXACT copy of the baseline architecture — do not modify the conv stack.

    Conv blocks, GroupNormalization(groups=8), GAP, dropouts (0.2 / 0.4) and
    Dense(64) head are identical to enhanced-gournetv2-12-mseed.ipynb so the
    comparison stays focused on the data pipeline, not the model.
    Only the *augment* block gains the RandomZoom + RandomContrast layers the
    baseline intro already claimed (its code only had flip + rotation).
    """
    rescale = tf.keras.Sequential([layers.Rescaling(1.0 / 255)], name="Sequential-1")

    augment = tf.keras.Sequential(
        [
            layers.RandomFlip("horizontal_and_vertical"),
            layers.RandomRotation(0.2),
            layers.RandomZoom(0.15),
            layers.RandomContrast(0.15),
        ],
        name="Sequential-2",
    )

    inputs = tf.keras.Input(shape=input_shape)
    x = rescale(inputs)
    x = augment(x)

    def conv_bn_block(x, filters, name_prefix):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name_prefix}_Conv")(x)
        x = layers.GroupNormalization(groups=8, name=f"{name_prefix}_GN")(x)
        x = layers.Activation("relu", name=f"{name_prefix}_ReLU")(x)
        x = layers.MaxPooling2D(pool_size=2, name=f"{name_prefix}_Pool")(x)
        return x

    x = conv_bn_block(x, 32, "Block1")
    x = conv_bn_block(x, 64, "Block2")
    x = conv_bn_block(x, 64, "Block3")
    x = conv_bn_block(x, 64, "Block4")

    x = layers.Dropout(0.2, name="Spatial_Dropout")(x)
    x = layers.GlobalAveragePooling2D(name="GAP")(x)

    x = layers.Dense(64, name="Dense-1")(x)
    x = layers.GroupNormalization(groups=8, name="Dense-1_GN")(x)
    x = layers.Activation("relu", name="Dense-1_ReLU")(x)
    x = layers.Dropout(0.4, name="Head_Dropout")(x)

    outputs = layers.Dense(num_classes, activation="softmax", name="Dense-2")(x)

    return models.Model(inputs, outputs, name="GourNet_v2")

model_v2 = build_gournet_v2(num_classes=len(class_names))
model_v2.summary()
print(f'\nGourNet v2 total parameters: {model_v2.count_params():,}')


## 6. Compile & train (same optimizer, loss, callbacks as baseline)


In [ ]:
model_v2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'],
)

early_stop_v2 = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=EARLY_STOP_PATIENCE, restore_best_weights=True
)
checkpoint_v2 = tf.keras.callbacks.ModelCheckpoint(
    str(WORKING / 'gournet_v2_robust_best.keras'), monitor='val_loss', save_best_only=True
)
reduce_lr_v2 = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6
)

history_v2 = model_v2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=[early_stop_v2, checkpoint_v2, reduce_lr_v2],
)


## 7. Evaluate + logit Grad-CAM sanity check

Grad-CAM uses pre-softmax logits at `Block4_ReLU` (the app fix). Softmax maps saturate near 1.0 and pre-norm `Block4_Conv` distorts space — both produced background-looking maps on confident images.


In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

test_loss_v2, test_acc_v2 = model_v2.evaluate(test_ds)
print(f'Robust v2 test accuracy: {test_acc_v2 * 100:.2f}%  loss: {test_loss_v2:.4f}')

y_true_v2, y_pred_v2 = [], []
for images, labels in test_ds:
    y_true_v2.extend(labels.numpy())
    y_pred_v2.extend(np.argmax(model_v2.predict(images, verbose=0), axis=1))
y_true_v2, y_pred_v2 = np.array(y_true_v2), np.array(y_pred_v2)
print(classification_report(y_true_v2, y_pred_v2, target_names=class_names, digits=4))
cm = confusion_matrix(y_true_v2, y_pred_v2)
plt.figure(figsize=(9, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('Robust GourNet v2 — Confusion Matrix (Test Set)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(WORKING / 'robust_confusion_matrix.png', dpi=150)
plt.show()


In [ ]:
def gradcam_logit(model, img_batch, layer_name='Block4_ReLU', pred_index=None):
    """Pre-softmax logit Grad-CAM. Returns normalized map; zeros = no positive signal."""
    head = model.layers[-1]
    probe = tf.keras.Model(model.inputs, [model.get_layer(layer_name).output, head.input])
    img_batch = tf.convert_to_tensor(img_batch, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(img_batch)
        conv_out, feats = probe(img_batch, training=False)
        logits = tf.matmul(feats, head.kernel) + head.bias
        if pred_index is None:
            pred_index = int(tf.argmax(logits[0]))
        score = logits[:, pred_index]
    grads = tape.gradient(score, conv_out)
    w = tf.reduce_mean(grads, axis=(1, 2))
    heat = tf.reduce_sum(conv_out[0] * w[0], axis=-1)
    heat = tf.maximum(heat, 0)
    return (heat / tf.reduce_max(heat)).numpy() if float(tf.reduce_max(heat)) > 0 else heat.numpy()


shown = 0
plt.figure(figsize=(12, 8))
for images, labels in test_ds.take(2):
    for i in range(images.shape[0]):
        if shown >= 6:
            break
        h = gradcam_logit(model_v2, images[i:i+1])
        shown += 1
        ax = plt.subplot(2, 3, shown)
        ax.imshow(images[i].numpy().astype('uint8'))
        ax.imshow(tf.image.resize(h[..., None], IMG_SIZE).numpy()[..., 0],
                  cmap='jet', alpha=0.45, vmin=0, vmax=1)
        ax.set_title(f'{class_names[int(labels[i])]}', fontsize=9)
        ax.axis('off')
plt.suptitle('Logit Grad-CAM at Block4_ReLU (not a segmentation mask)')
plt.tight_layout()
plt.savefig(WORKING / 'robust_gradcam_check.png', dpi=130)
plt.show()


## 8. Background-robustness probes (the actual test for this fix)

Same leaves, randomized paper backgrounds: a background-reliant model drops. Background-only images should score ≈chance (12.5%); well above chance means the background alone predicts the class.


In [ ]:
def _eval_triplets(df, model, seed, n_bg=1):
    rng = np.random.default_rng(seed)
    y_true, p_orig, p_swap, p_bgonly, p_leafonly = [], [], [], [], []
    for _, row in df.iterrows():
        img = np.array(Image.open(row['filepath']).convert('RGB').resize((224, 224)), dtype=np.float32)
        m = np.array(Image.open(MASK_LOOKUP[row['filepath']]).convert('L').resize((224, 224)),
                   dtype=np.float32)[..., None] / 255.0
        bg = _random_paper_background(224, 224, rng)
        swapped = (m * img + (1 - m) * bg)[None]
        gray = np.full_like(img, 128.0)[None]
        leafonly = (m * img + (1 - m) * 128.0)[None]
        y = CLASS_TO_IDX[row['class']]
        y_true.append(y)
        p_orig.append(model.predict(img[None], verbose=0)[0])
        p_swap.append(model.predict(swapped, verbose=0)[0])
        p_bgonly.append(model.predict(bg[None].astype(np.float32), verbose=0)[0])
        p_leafonly.append(model.predict(leafonly.astype(np.float32), verbose=0)[0])
    out = {}
    for k, p in [('orig', p_orig), ('swap', p_swap), ('bgonly', p_bgonly), ('leafonly', p_leafonly)]:
        out[k] = float(np.mean(np.argmax(np.array(p), axis=1) == np.array(y_true)))
    out['swap_drop'] = out['orig'] - out['swap']
    return out


probes = _eval_triplets(test_df, model_v2, SEED)
print('Robustness probes (test split):')
for k, v in probes.items():
    print(f'  {k}: {v*100:.2f}%')
print('Interpretation: small swap_drop + bgonly near 12.5% = background-independent. ' +
      'Large drop or high bgonly = shortcut remains.')


## 9. Multi-seed run — same 12 seeds, fresh splits + fresh weights per seed

Masks are reused (image property); splits, background draws, init, and shuffle are re-seeded. Saves `robust_multiseed_results.csv` for direct comparison with the baseline CSV.


In [ ]:
def run_robust_seed(seed: int, files_df: pd.DataFrame) -> dict:
    tf.keras.utils.set_random_seed(seed)
    tr_df, va_df, te_df = stratified_file_split(files_df, seed)
    tr_ds = train_ds_from_df(tr_df, seed)
    va_ds = ds_from_df(va_df, shuffle=False, seed=seed)
    te_ds = ds_from_df(te_df, shuffle=False, seed=seed)
    m = build_gournet_v2(num_classes=len(CLASS_NAMES))
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])
    es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=EARLY_STOP_PATIENCE,
                                          restore_best_weights=True)
    rl = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
    hist = m.fit(tr_ds, validation_data=va_ds, epochs=MAX_EPOCHS, callbacks=[es, rl], verbose=0)
    te_loss, te_acc = m.evaluate(te_ds, verbose=0)
    probes = _eval_triplets(te_df, m, seed)
    return {'seed': seed, 'test_accuracy': float(te_acc), 'test_loss': float(te_loss),
            'epochs_trained': len(hist.history['loss']), **{f'probe_{k}': v for k, v in probes.items()}}


multiseed_results = []
for s in SEEDS:
    print(f'--- seed={s} ---')
    r = run_robust_seed(s, FILES_DF)
    multiseed_results.append(r)
    print(f"  acc={r['test_accuracy']*100:.2f}% loss={r['test_loss']:.4f} epochs={r['epochs_trained']} " +
          f"swap_drop={r['probe_swap_drop']*100:.2f}pp bgonly={r['probe_bgonly']*100:.2f}%")


In [ ]:
results_df = pd.DataFrame(multiseed_results)
results_df['test_accuracy_pct'] = results_df['test_accuracy'] * 100
print(results_df[['seed', 'test_accuracy_pct', 'test_loss', 'epochs_trained',
                  'probe_swap_drop', 'probe_bgonly']].to_string(index=False))
print(f"\nMean acc: {results_df['test_accuracy_pct'].mean():.2f}% ± {results_df['test_accuracy_pct'].std(ddof=1):.2f}%")
print(f"Mean swap_drop: {results_df['probe_swap_drop'].mean()*100:.2f}pp | " +
      f"Mean bgonly: {results_df['probe_bgonly'].mean()*100:.2f}% (chance=12.5%)")
results_df.to_csv(WORKING / 'robust_multiseed_results.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].bar([str(r['seed']) for r in multiseed_results], results_df['test_accuracy_pct'])
axes[0].axhline(97.0, color='tab:red', linestyle='--', label='Paper 97%')
axes[0].set_title('Test accuracy per seed')
axes[0].legend()
axes[1].bar([str(r['seed']) for r in multiseed_results], results_df['probe_swap_drop'] * 100, color='tab:orange')
axes[1].set_title('Background-swap drop (pp) — lower is better')
axes[2].bar([str(r['seed']) for r in multiseed_results], results_df['probe_bgonly'] * 100, color='tab:green')
axes[2].axhline(12.5, color='black', linestyle='--', label='Chance')
axes[2].set_title('Background-only acc — near chance is better')
axes[2].legend()
plt.tight_layout()
plt.savefig(WORKING / 'robust_multiseed.png', dpi=150)
plt.show()


### Reading the result (thesis-safe wording)

- Claim the fix **only if**: test-acc mean holds near baseline **and** swap_drop is small **and** bgonly is near chance across seeds.
- Do not claim Grad-CAM overlays prove symptom grounding — they are 28×28 upsampled explanations, not segmentations.
- If swap_drop stays large or bgonly stays high, the shortcut survives: next step is better masks (manual review of CuttingWeevil stems especially), not attention modules or heatmap masking.
- Export: save the best seed's model as `gournet_v2_robust_best.keras` and point the app at it. The app already explains `Block4_ReLU` logits, matching Section 7.


In [ ]:
model_v2.save(str(WORKING / 'gournet_v2_robust_model.keras'))
print('Saved robust model to', WORKING / 'gournet_v2_robust_model.keras')
